In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from torch.utils.data import Dataset, DataLoader

In [2]:
df = pd.read_csv("data/titanic/train.csv")
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [3]:
features = ["Pclass","Sex","Age","SibSp","Parch","Fare","Embarked"]
df = df[features + ["Survived"]]

# missing values
df["Age"] = df["Age"].fillna(df["Age"].median())
df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])

# encode categoricals
df["Sex"] = LabelEncoder().fit_transform(df["Sex"])
df["Embarked"] = LabelEncoder().fit_transform(df["Embarked"])

X = df.drop("Survived", axis=1).values
y = df["Survived"].values


In [4]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)


In [5]:
class TitanicDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


In [6]:
class TitanicDatasetTransform(Dataset):
    def __init__(self, X, y, transform=None):
        self.X = X
        self.y = y
        self.transform = transform

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        x = self.X[idx]
        y = self.y[idx]

        if self.transform:
            x = self.transform(x)

        return torch.tensor(x, dtype=torch.float32), torch.tensor(y, dtype=torch.float32)


In [7]:
def noise_transform(x):
    noise = np.random.normal(0, 0.01, size=x.shape)
    return x + noise

In [ ]:
train_ds = TitanicDatasetTransform(X_train, y_train, transform=noise_transform)
val_ds = TitanicDataset(X_val, y_val)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=32)

In [9]:
class TitanicNN(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        
        self.net = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.BatchNorm1d(32),
            nn.Dropout(0.3),
            
            nn.Linear(32, 16),
            nn.ReLU(),
            
            nn.Linear(16, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x)


In [10]:
model = TitanicNN(input_dim=X_train.shape[1])

criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [11]:
def train_model(model, train_loader, val_loader, epochs=100, patience=10):
    
    best_val_loss = float("inf")
    patience_counter = 0
    
    for epoch in range(epochs):
        
        # ---- TRAIN ----
        model.train()
        train_loss = 0
        
        for Xb, yb in train_loader:
            preds = model(Xb).squeeze()
            loss = criterion(preds, yb)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
        
        train_loss /= len(train_loader)
        
        # ---- VALIDATION ----
        model.eval()
        val_loss = 0
        correct = 0
        total = 0
        
        with torch.no_grad():
            for Xb, yb in val_loader:
                preds = model(Xb).squeeze()
                loss = criterion(preds, yb)
                val_loss += loss.item()
                
                pred_labels = (preds > 0.5).float()
                correct += (pred_labels == yb).sum().item()
                total += len(yb)
        
        val_loss /= len(val_loader)
        val_acc = correct / total
        
        print(f"Epoch {epoch+1}: train_loss={train_loss:.4f} val_loss={val_loss:.4f} val_acc={val_acc:.4f}")
        
        # ---- EARLY STOPPING ----
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            best_weights = model.state_dict()
        else:
            patience_counter += 1
        
        if patience_counter >= patience:
            print("Early stopping triggered")
            break
    
    model.load_state_dict(best_weights)
    return model


In [12]:
model = train_model(model, train_loader, val_loader)


Epoch 1: train_loss=0.6556 val_loss=0.6366 val_acc=0.7095
Epoch 2: train_loss=0.5941 val_loss=0.5800 val_acc=0.7598
Epoch 3: train_loss=0.5401 val_loss=0.5343 val_acc=0.7598
Epoch 4: train_loss=0.4838 val_loss=0.4996 val_acc=0.7709
Epoch 5: train_loss=0.4833 val_loss=0.4812 val_acc=0.7933
Epoch 6: train_loss=0.4545 val_loss=0.4714 val_acc=0.7989
Epoch 7: train_loss=0.4619 val_loss=0.4651 val_acc=0.7877
Epoch 8: train_loss=0.4560 val_loss=0.4627 val_acc=0.7821
Epoch 9: train_loss=0.4454 val_loss=0.4594 val_acc=0.7877
Epoch 10: train_loss=0.4659 val_loss=0.4582 val_acc=0.7821
Epoch 11: train_loss=0.4370 val_loss=0.4571 val_acc=0.7821
Epoch 12: train_loss=0.4341 val_loss=0.4561 val_acc=0.7877
Epoch 13: train_loss=0.4506 val_loss=0.4532 val_acc=0.7877
Epoch 14: train_loss=0.4124 val_loss=0.4521 val_acc=0.7877
Epoch 15: train_loss=0.4115 val_loss=0.4513 val_acc=0.7821
Epoch 16: train_loss=0.4064 val_loss=0.4518 val_acc=0.7821
Epoch 17: train_loss=0.4229 val_loss=0.4517 val_acc=0.7877
Epoch 

In [13]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for Xb, yb in val_loader:
        preds = model(Xb).squeeze()
        pred_labels = (preds > 0.5).float()
        correct += (pred_labels == yb).sum().item()
        total += len(yb)

print("Validation Accuracy:", correct/total)


Validation Accuracy: 0.7988826815642458
